# US iOS keyword-product deep-dive audit

## TL;DR

This notebook validates the eight-candidate comparison, the explicit gate-first research order, and the Runway five-second cost examples used in the report. It deliberately does not calculate Apple search volume, revenue, success probability, or a v4 opportunity score.

## Method boundary

The 1-5 fields are transparent desk-research heuristics from public US App Store listings, pricing, visible non-random reviews, technical scope and official policy documents. Research priority is an explicit decision sequence, not the output of a composite formula.

In [1]:
import json
from pathlib import Path

DATA_PATH = Path('../data/candidate_analysis.json')
data = json.loads(DATA_PATH.read_text())
rows = sorted(data['candidates'], key=lambda row: row['research_priority'])

assert len(rows) == 8
assert [row['research_priority'] for row in rows] == list(range(1, 9))
assert len({row['candidate_id'] for row in rows}) == len(rows)
assert data['apple_exact_volume_status'] == 'unavailable_without_apple_ads_search_popularity_or_search_term_export'

score_fields = ('public_demand_proxy', 'purchase_intent_proxy', 'competition', 'solo_readiness', 'founder_style_fit')
assert all(1 <= row[field] <= 5 for row in rows for field in score_fields)
assert all(row['evidence_grade'] in {'A', 'B', 'C', 'D'} for row in rows)
assert rows[0]['candidate_id'] == 'before-after-reel'
assert rows[-1]['candidate_id'] == 'batch-watermark'

passport = next(row for row in rows if row['candidate_id'] == 'us-baby-passport-coach')
assert 'Background removal' in passport['mvp']['excluded']
assert 'acceptance guarantee' in passport['mvp']['excluded']
memory = next(row for row in rows if row['candidate_id'] == 'memorymotion')
assert memory['research_action'] == 'TECHNICAL_RAT'
assert memory['solo_readiness'] == 2

credit_usd = 0.01
seconds = 5
runway_credits_per_second = {'Gen-4 Turbo': 5, 'Gen-4.5': 12}
runway_costs = {
    model: rate * seconds * credit_usd
    for model, rate in runway_credits_per_second.items()
}
assert runway_costs == {'Gen-4 Turbo': 0.25, 'Gen-4.5': 0.60}
assert runway_costs['Gen-4.5'] * 2 == 1.20

print('priority | candidate | demand | purchase | competition | solo | style | action')
print('--- | --- | --- | --- | --- | --- | --- | ---')
for row in rows:
    print(' | '.join(map(str, (
        row['research_priority'], row['product_name'], row['public_demand_proxy'],
        row['purchase_intent_proxy'], row['competition'], row['solo_readiness'],
        row['founder_style_fit'], row['research_action']
    ))))

print('\nRunway public list-price examples for one 5-second generation:')
for model, cost in runway_costs.items():
    print(f'- {model}: ${cost:.2f}; one full retry: ${cost * 2:.2f}')

print('\nPASS: candidate count, ordering, bounds, compliance scope, AI risk and cost arithmetic')
print('LIMIT: no Apple Ads Search Popularity or search-term export; no confirmed keyword volume')


priority | candidate | demand | purchase | competition | solo | style | action
--- | --- | --- | --- | --- | --- | --- | ---
1 | Before/After Reel | 3 | 5 | 2 | 5 | 5 | USER_BEHAVIOR_TEST
2 | Exact-MB Video Compressor | 4 | 4 | 4 | 4 | 3 | TECHNICAL_RAT
3 | Birthday / Occasion Reel | 3 | 4 | 2 | 5 | 5 | TASK_AND_PAYMENT_TEST
4 | Family Photo Colorizer | 3 | 5 | 3 | 4 | 5 | TECHNICAL_RAT
5 | Family Archive Scanner | 4 | 4 | 4 | 3 | 4 | WEDGE_PAYMENT_TEST
6 | US Baby Passport Capture Coach | 4 | 5 | 5 | 3 | 3 | COMPLIANCE_RAT
7 | MemoryMotion | 4 | 5 | 4 | 2 | 5 | TECHNICAL_RAT
8 | Batch Watermark for Marketplace Sellers | 4 | 4 | 4 | 5 | 2 | PARK_PREFERENCE_MISMATCH

Runway public list-price examples for one 5-second generation:
- Gen-4 Turbo: $0.25; one full retry: $0.50
- Gen-4.5: $0.60; one full retry: $1.20

PASS: candidate count, ordering, bounds, compliance scope, AI risk and cost arithmetic
LIMIT: no Apple Ads Search Popularity or search-term export; no confirmed keyword volume


## Interpretation

- The order is a research-investment sequence, not a market-size leaderboard.
- Before/After Reel leads because its decisive user test is cheap and its ongoing operations are bounded.
- Exact-MB Compressor needs a codec corpus RAT before UI investment.
- Old Photo Colorizer is the lower-operations AI option if an on-device model preserves geometry.
- MemoryMotion has the strongest visual upside and the weakest solo readiness because retries, identity, privacy, credits and support compound.